# Truc quan hoa ket qua (Chuan web Colab)

Notebook nay chay theo thu tu tu tren xuong trong Colab:
1. Clone/cap nhat repo
2. Cai thu vien
3. Upload kaggle.json
4. Chay pipeline
5. Doc va ve ket qua RFM + MBA

In [ ]:
from pathlib import Path
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 1) Clone hoac cap nhat repo

In [ ]:
REPO_URL = 'https://github.com/phoudsavanhKongmany/final-bigdata-project-nhom9Test.git'
REPO_NAME = 'final-bigdata-project-nhom9Test'
REPO_DIR = Path('/content') / REPO_NAME

if not REPO_DIR.exists():
    print('Dang clone repo...')
    !git clone {REPO_URL}
else:
    print('Repo da ton tai, dang cap nhat...')
    !git -C {REPO_DIR} pull

%cd /content/final-bigdata-project-nhom9Test
print('Thu muc hien tai:', Path.cwd().resolve())

## 2) Cai thu vien

In [ ]:
!pip -q install -r requirements.txt

## 3) Upload kaggle.json
Tai file `kaggle.json` tu tai khoan Kaggle, roi chay cell duoi de upload.

In [ ]:
from google.colab import files

uploaded = files.upload()

if 'kaggle.json' not in uploaded:
    raise RuntimeError('Ban can upload tep kaggle.json de tiep tuc.')

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
print('Da cau hinh Kaggle credentials thanh cong.')

## 4) Chay pipeline de tao du lieu

In [ ]:
!python3 src/main_pipeline.py --project both --step all --mba-sample-fraction 0.2 --min-support 0.01 --min-confidence 0.2

## 5) Kiem tra duong dan output

In [ ]:
ROOT = Path('/content/final-bigdata-project-nhom9Test')
RFM_SEGMENT_PATH = ROOT / 'data/3_curated/results/rfm/customer_segments'
RFM_SUMMARY_PATH = ROOT / 'data/3_curated/results/rfm/segment_summary'
MBA_RULES_PATH = ROOT / 'data/3_curated/results/mba/association_rules'

print('ROOT:', ROOT)
print('Co tep phan khuc RFM:', RFM_SEGMENT_PATH.exists())
print('Co tep tong hop RFM:', RFM_SUMMARY_PATH.exists())
print('Co tep luat MBA:', MBA_RULES_PATH.exists())

## 6) Doc ket qua RFM

In [ ]:
rfm_segments = pd.read_parquet(RFM_SEGMENT_PATH)
rfm_summary = pd.read_parquet(RFM_SUMMARY_PATH)

display(rfm_segments.head())

if 'business_score' in rfm_summary.columns:
    display(rfm_summary.sort_values('business_score', ascending=False))
elif 'avg_monetary' in rfm_summary.columns:
    display(rfm_summary.sort_values('avg_monetary', ascending=False))
else:
    display(rfm_summary)

## 7) Bieu do RFM

In [ ]:
bang_dem = (
    rfm_segments['segment_label']
    .value_counts()
    .rename_axis('segment_label')
    .reset_index(name='customers')
)

ax = sns.barplot(
    data=bang_dem,
    x='segment_label',
    y='customers',
    hue='segment_label',
    palette='Set2',
    legend=False
)
ax.set_title('So luong khach hang theo phan khuc')
ax.set_xlabel('Phan khuc')
ax.set_ylabel('So khach hang')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
ds_chi_so = [c for c in ['avg_recency_days', 'avg_frequency', 'avg_monetary'] if c in rfm_summary.columns]

if ds_chi_so and 'segment_label' in rfm_summary.columns:
    metrics_long = rfm_summary.melt(
        id_vars=['segment_label'],
        value_vars=ds_chi_so,
        var_name='chi_so',
        value_name='gia_tri'
    )

    g = sns.catplot(
        data=metrics_long,
        x='segment_label',
        y='gia_tri',
        col='chi_so',
        kind='bar',
        sharey=False,
        palette='Set3',
        height=4,
        aspect=1.1
    )
    g.set_titles('{col_name}')
    for truc in g.axes.flat:
        truc.tick_params(axis='x', rotation=20)
    plt.tight_layout()
    plt.show()
else:
    print('Khong du cot de ve bieu do chi so RFM.')

## 8) Doc va ve luat MBA

In [ ]:
mba_rules = pd.read_parquet(MBA_RULES_PATH)

if mba_rules.empty:
    print('Tep luat ket hop ton tai nhung rong.')
else:
    top_rules = mba_rules.sort_values('confidence', ascending=False).head(15).copy()

    if 'antecedent' in top_rules.columns and 'consequent' in top_rules.columns:
        top_rules['rule'] = top_rules['antecedent'].astype(str) + ' => ' + top_rules['consequent'].astype(str)
    elif 'antecedents' in top_rules.columns and 'consequents' in top_rules.columns:
        top_rules['rule'] = top_rules['antecedents'].astype(str) + ' => ' + top_rules['consequents'].astype(str)
    else:
        top_rules['rule'] = top_rules.index.astype(str)

    cols_hien = [c for c in ['rule', 'confidence', 'lift', 'support'] if c in top_rules.columns]
    display(top_rules[cols_hien])

    plt.figure(figsize=(12, 7))
    sns.barplot(
        data=top_rules,
        x='confidence',
        y='rule',
        hue='rule',
        palette='viridis',
        legend=False
    )
    plt.title('Top 15 luat ket hop theo do tin cay')
    plt.xlabel('Do tin cay')
    plt.ylabel('Luat')
    plt.tight_layout()
    plt.show()